In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

warnings.filterwarnings('ignore')

CLASS_MAP = {
    'normal'                        : 'Normal',
    'anomalous(DoSattack)'          : 'DoS',
    'anomalous(scan)'               : 'Scan',
    'anomalous(malitiousControl)'   : 'MaliciousControl',
    'anomalous(malitiousOperation)' : 'MaliciousOperation',
    'anomalous(spying)'             : 'Spying',
    'anomalous(dataProbing)'        : 'DataProbing',
    'anomalous(wrongSetUp)'         : 'WrongSetUp',
}
CLASS_TO_INT = {v: i for i, v in enumerate(CLASS_MAP.values())}
INT_TO_CLASS = {v: k for k, v in CLASS_TO_INT.items()}

SEARCH_PATHS = [
    Path("DS2OS.csv"),
    Path.home() / "Downloads" / "DS2OS.csv",
    (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv"),
]

CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError("DS2OS.csv no encontrado.")

df_raw = pd.read_csv(CSV_PATH)
print(f"Shape: {df_raw.shape}")

Shape: (357952, 13)


In [2]:
df = df_raw.copy()

df['accessedNodeType'] = df['accessedNodeType'].fillna('Malicious')
df['value'] = df['value'].replace({'False': 0, 'True': 1, 'Twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)
df = df.drop(columns=['timestamp'])
df['y'] = df['normality'].map(CLASS_MAP).map(CLASS_TO_INT).astype(np.int8)
df = df.drop(columns=['normality'])

CAT_COLS = [
    'sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
    'destinationServiceAddress', 'destinationServiceType',
    'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation',
]
for col in CAT_COLS:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

FEATURE_COLS = CAT_COLS + ['value']
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['y'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

CLASSES     = sorted(np.unique(y_train))
CLASS_NAMES = INT_TO_CLASS

print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {len(FEATURE_COLS)}")

Train: 286,361 | Test: 71,591 | Features: 11


In [3]:
class KDENaiveBayes(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, bandwidth=0.5):
        self.bandwidth = bandwidth

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.classes_ = np.unique(y)
        self.kdes_, self.priors_ = {}, {}
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / len(X)
            self.kdes_[c] = KernelDensity(bandwidth=self.bandwidth).fit(X_c)
        return self

    def predict_proba(self, X):
        X = np.array(X)
        log_p = np.array([
            np.log(self.priors_[c]) + self.kdes_[c].score_samples(X)
            for c in self.classes_
        ]).T
        log_p -= log_p.max(axis=1, keepdims=True)
        p = np.exp(log_p)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


classifiers = {
    'A1 - J48'          : DecisionTreeClassifier(criterion='entropy', min_samples_leaf=2, random_state=1),
    'A2 - Meta Pagging' : BaggingClassifier(estimator=DecisionTreeClassifier(min_samples_leaf=2, random_state=1), n_estimators=10, max_samples=1.0, random_state=1),
    'A3 - RandomTree'   : DecisionTreeClassifier(max_features=6, min_samples_leaf=1, random_state=1),
    'A4 - REPTree'      : DecisionTreeClassifier(min_samples_leaf=2, random_state=1),
    'A5 - AdaBoostM1'   : AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=1), n_estimators=10, random_state=1, algorithm='SAMME'),
    'A6 - DecisionStump': DecisionTreeClassifier(max_depth=1, random_state=1),
    'A7 - NaiveBayes'   : KDENaiveBayes(bandwidth=0.5),
    'A8 - SVM'          : SVC(kernel='rbf', probability=True, C=1.0, gamma='scale', random_state=1),
}

trained_clfs = {}
for name, clf in classifiers.items():
    print(f"Entrenando {name}...", end=' ', flush=True)
    clf.fit(X_train, y_train)
    trained_clfs[name] = clf
    print("listo.")

Entrenando A1 - J48... listo.
Entrenando A2 - Meta Pagging... listo.
Entrenando A3 - RandomTree... listo.
Entrenando A4 - REPTree... listo.
Entrenando A5 - AdaBoostM1... listo.
Entrenando A6 - DecisionStump... listo.
Entrenando A7 - NaiveBayes... listo.
Entrenando A8 - SVM... listo.


In [4]:
def compute_weka_metrics(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    tp_rates, fp_rates, supports = [], [], []
    for i in range(len(classes)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FP - FN
        support = cm[i, :].sum()
        tp_rates.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        fp_rates.append(FP / (FP + TN) if (FP + TN) > 0 else 0.0)
        supports.append(support)
    total = sum(supports)
    tp_w = sum(tp * s for tp, s in zip(tp_rates, supports)) / total
    fp_w = sum(fp * s for fp, s in zip(fp_rates, supports)) / total
    return tp_w, fp_w


SEP = '=' * 62
print("Voting Ensemble — Dataset DS2OS")
print(SEP)
print(f"{'Clasificador':<22} {'TP Rate':>8} {'FP Rate':>8} {'Accuracy':>10}")
print('-' * 62)

all_probas = []
for name, clf in trained_clfs.items():
    y_pred = clf.predict(X_test)
    acc    = accuracy_score(y_test, y_pred) * 100
    tp, fp = compute_weka_metrics(y_test, y_pred, CLASSES)
    print(f"{name:<22} {tp:>8.3f} {fp:>8.3f} {acc:>9.2f}%")
    if hasattr(clf, 'predict_proba'):
        p = clf.predict_proba(X_test)
        clf_classes = list(clf.classes_)
        p_full = np.zeros((len(X_test), len(CLASSES)))
        for j, c in enumerate(CLASSES):
            if c in clf_classes:
                p_full[:, j] = p[:, clf_classes.index(c)]
        all_probas.append(p_full)

print('-' * 62)
avg_proba  = np.mean(all_probas, axis=0)
y_pred_ens = np.array(CLASSES)[np.argmax(avg_proba, axis=1)]
acc_e      = accuracy_score(y_test, y_pred_ens) * 100
tp_e, fp_e = compute_weka_metrics(y_test, y_pred_ens, CLASSES)
print(f"{'E  - Proposed Model':<22} {tp_e:>8.3f} {fp_e:>8.3f} {acc_e:>9.2f}%")
print(SEP)

Voting Ensemble — Dataset DS2OS
Clasificador            TP Rate  FP Rate   Accuracy
--------------------------------------------------------------
A1 - J48                  0.995    0.181     99.46%
A2 - Meta Pagging         0.995    0.181     99.46%
A3 - RandomTree           0.995    0.181     99.46%
A4 - REPTree              0.995    0.181     99.46%
A5 - AdaBoostM1           0.977    0.794     97.71%
A6 - DecisionStump        0.977    0.794     97.71%
A7 - NaiveBayes           0.992    0.256     99.21%
A8 - SVM                  0.995    0.181     99.46%
--------------------------------------------------------------
E  - Proposed Model       0.995    0.181     99.46%
